# Module 2 · Transformers et embeddings — Travaux pratiques
**Togo AI Lab · Formation IA · Version IT** · Durée : environ 1 h 30

**Ce que vous allez faire :** donner du *sens* à des textes. Au module 1, la machine comptait les mots. Ici, elle va placer chaque phrase sur une « carte du sens » (un *embedding*) pour retrouver, parmi 16 fiches d'information, celle qui répond à la question d'un usager, même quand les mots sont différents.

**À la fin, vous saurez :**
1. transformer des phrases en embeddings et mesurer leur proximité ;
2. construire une recherche sémantique et la comparer à une recherche par mots ;
3. visualiser et regrouper des textes selon leur sens ;
4. observer la tokenisation en sous-mots et son effet sur les fautes et les langues locales ;
5. décider quand répondre « je ne sais pas » grâce à un seuil.

**Comment utiliser ce notebook (Google Colab)**
- Une cellule de code s'exécute avec **Maj + Entrée**, dans l'ordre. La première exécution télécharge un modèle (quelques minutes) : internet est nécessaire.
- Les cellules **👉 À vous** contiennent des `____` à remplacer. Les consignes et indices sont juste au-dessus.
- Un GPU n'est pas nécessaire. Toutes les données sont **fictives**.

## 0 · Préparation
Installation et chargement des outils.

In [ ]:
try:
    import sentence_transformers
except ImportError:
    %pip install -q sentence-transformers

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

pd.set_option("display.max_colwidth", 80)
print("Outils prêts ✔")

## 1 · Les données
16 fiches d'information fictives d'un portail de services publics, et 9 questions d'usagers. **Exécutez simplement la cellule.**

In [ ]:
# Données fictives
FICHES = [
    ('Calendrier vaccinal des nourrissons', 'Les vaccins recommandés de la naissance à deux ans, et les dates où les administrer au centre de santé.', 'sante'),
    ('Établir un passeport', 'Pièces à fournir et étapes pour obtenir un document de voyage international.', 'administration'),
    ('Inscrire son enfant à l’école primaire', 'Dates d’inscription, documents demandés et liste des établissements de la commune.', 'education'),
    ('Déclarer une naissance', 'Démarches à la mairie dans les jours qui suivent l’arrivée d’un enfant.', 'administration'),
    ('Se protéger du paludisme', 'Moustiquaires imprégnées, assainissement autour de la maison, signes qui doivent alerter.', 'sante'),
    ('Demander une bourse d’études', 'Conditions, dossier à constituer et calendrier de l’aide financière pour les étudiants.', 'education'),
    ('Horaires du centre de santé', 'Jours et heures de consultation, urgences et permanence du week-end.', 'sante'),
    ('Renouveler sa carte d’identité', 'Documents à présenter et délais pour obtenir une nouvelle carte.', 'administration'),
    ('Consultations prénatales', 'Suivi de la grossesse au centre de santé : visites, examens et conseils pour la future maman.', 'sante'),
    ('Obtenir un acte de mariage', 'Copie de son acte de mariage auprès de la mairie du lieu de célébration.', 'administration'),
    ('Calendrier scolaire', 'Dates de la rentrée, des congés et des examens de l’année scolaire.', 'education'),
    ('Cantine scolaire', 'Inscription aux repas de l’école et tarifs selon la situation de la famille.', 'education'),
    ('Don du sang', 'Conditions pour donner son sang et lieux de collecte organisés dans la région.', 'sante'),
    ('Permis de conduire', 'Pièces à fournir pour passer l’examen du permis et délais pour obtenir le document.', 'administration'),
    ('Soigner la diarrhée chez l’enfant', 'Réhydratation à la maison et signes pour consulter sans attendre.', 'sante'),
    ('Examens et résultats du baccalauréat', 'Dates des épreuves, consultation des résultats et demande de relevé de notes.', 'education'),
]

# (question, numéro de la bonne fiche ; None = aucune fiche ne convient)
QUESTIONS = [
    ('Où faire piquer mon petit dernier pour le protéger des maladies ?', 0),
    ('Mon fils a besoin d’un papier pour voyager à l’étranger.', 1),
    ('Que faire contre les moustiques qui rendent les gens malades ?', 4),
    ('J’ai eu un bébé hier, que dois-je faire à la mairie ?', 3),
    ('Une aide financière pour payer les études de ma fille.', 5),
    ('À quelle heure puis-je voir l’infirmier ?', 6),
    ('Mon bébé a le ventre qui coule, que faire ?', 14),
    ('Quand sortent les notes du bac ?', 15),
    ('Comment obtenir un prêt pour acheter une voiture ?', None),
]

fiches = pd.DataFrame(FICHES, columns=["titre", "texte", "theme"])
fiches["document"] = fiches["titre"] + ". " + fiches["texte"]
print(len(fiches), "fiches,", len(QUESTIONS), "questions")
fiches[["titre", "theme"]].head(8)

## 2 · Des phrases aux embeddings
Un *modèle d'embeddings* transforme une phrase en une liste de nombres : son « adresse » sur la carte du sens. Nous utilisons un modèle **multilingue** léger.

**👉 À vous — Exercice 1.** Encodez les trois phrases ci-dessous et affichez la taille du résultat.  
*Indice : `modele.encode(liste, normalize_embeddings=True)` ; la forme se lit avec `.shape` (nombre de phrases, nombre de nombres par phrase).*

In [ ]:
modele = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
phrases = ["Où faire vacciner mon bébé ?",
           "Calendrier des vaccins du nourrisson",
           "Comment renouveler mon passeport ?"]

In [ ]:
vecteurs = modele.encode(____, normalize_embeddings=____)
print(vecteurs.shape)

## 3 · Mesurer la proximité
Quand les vecteurs sont *normalisés*, un simple produit (`@`) donne une **similarité** : plus elle est proche de 1, plus les phrases ont des sens voisins.

**👉 À vous — Exercice 2.** Calculez la matrice de similarités entre les trois phrases.  
*Indice : `vecteurs @ vecteurs.T`.* Observez : quelles phrases sont les plus proches ? Est-ce attendu ?

In [ ]:
sim = ____ @ ____.T
pd.DataFrame(sim.round(2), index=phrases, columns=["P1", "P2", "P3"])

## 4 · Encoder tout le corpus
On encode **une seule fois** les 16 fiches. Seules les questions seront encodées à la volée.

In [ ]:
emb_fiches = modele.encode(fiches["document"].tolist(), normalize_embeddings=True)
print(emb_fiches.shape)

## 5 · La recherche sémantique
Pour une question : l'encoder, calculer sa similarité avec chaque fiche, garder les `k` meilleures.

**👉 À vous — Exercice 3.** Complétez la fonction `chercher`.  
*Indice : `emb_fiches @ vecteur_question` donne un score par fiche ; `np.argsort(scores)[::-1]` classe du meilleur au moins bon.*

In [ ]:
def chercher(question, k=3):
    v = modele.encode([question], normalize_embeddings=True)[0]
    scores = emb_fiches @ ____
    ordre = np.argsort(____)[::-1][:k]
    return [(int(i), float(scores[i])) for i in ordre]

chercher("Mon fils a besoin d'un papier pour voyager à l'étranger.")

In [ ]:
def afficher(question, k=3):
    print("Q :", question)
    for i, s in chercher(question, k):
        print(f"   {s:.2f}  {fiches['titre'][i]}")

afficher(QUESTIONS[0][0])
afficher(QUESTIONS[2][0])

## 6 · Comparer avec la recherche par mots
Reprenons la méthode du module 1 (TF-IDF) comme **référence** : elle cherche des mots communs entre la question et les fiches.

In [ ]:
tfidf = TfidfVectorizer(analyzer="word", lowercase=True)
X_fiches = tfidf.fit_transform(fiches["document"])

def chercher_mots(question, k=3):
    q = tfidf.transform([question])
    scores = (X_fiches @ q.T).toarray().ravel()
    ordre = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i])) for i in ordre]

def taux_top1(fonction):
    bons = [fonction(q)[0][0] == bonne for q, bonne in QUESTIONS if bonne is not None]
    return sum(bons), len(bons)

lignes = []
for q, bonne in QUESTIONS:
    if bonne is None:
        continue
    lignes.append({"question": q,
                   "attendue": fiches["titre"][bonne],
                   "mots": fiches["titre"][chercher_mots(q)[0][0]],
                   "sens": fiches["titre"][chercher(q)[0][0]]})
pd.DataFrame(lignes)

**👉 À vous — Exercice 4.** Calculez combien de questions (sur 8) chaque méthode place en tête.  
*Indice : la fonction `taux_top1(chercher_mots)` renvoie `(bonnes, total)`.* Que remarquez-vous sur les questions où la méthode par mots échoue ?

In [ ]:
print("par mots :", taux_top1(____))
print("par sens :", taux_top1(____))

## 7 · Savoir dire « je ne sais pas »
La dernière question (le prêt pour une voiture) n'a **aucune** fiche correspondante. Une recherche renvoie pourtant toujours un meilleur résultat. La parade : un **seuil** de similarité.

**👉 À vous — Exercice 5.** Affichez le meilleur score de chaque question, puis choisissez un seuil qui sépare les bonnes questions de la question hors sujet.  
*Indice : `chercher(q, k=1)[0][1]` donne le meilleur score. Le seuil dépend du modèle : il se choisit en regardant les valeurs.*

In [ ]:
for q, bonne in QUESTIONS:
    meilleur = chercher(q, k=1)[0][1]
    print(f"{meilleur:.2f}  {'(hors sujet)' if bonne is None else ''}  {q}")

SEUIL = ____  # à choisir

In [ ]:
def repondre(question, seuil=None):
    seuil = SEUIL if seuil is None else seuil
    i, score = chercher(question, k=1)[0]
    if score < seuil:
        return "Je ne sais pas : aucune fiche ne convient. Je vous oriente vers un agent."
    return f"Fiche proposée : « {fiches['titre'][i]} » (score {score:.2f})"

for q, _ in QUESTIONS[-3:]:
    print(q, "→", repondre(q))

> **À retenir.** Un seuil n'est jamais parfait : trop haut, on refuse de bonnes questions ; trop bas, on invente. On le règle sur des exemples réels, et on prévoit toujours un passage par un humain.

## 8 · Voir la carte du sens
Nos embeddings ont plusieurs centaines de nombres. Pour les **voir**, on les projette sur un plan (2 nombres) avec une ACP.

**👉 À vous — Exercice 6.** Projetez les fiches en 2D.  
*Indice : `PCA(n_components=2).fit_transform(emb_fiches)`.*

In [ ]:
points = PCA(n_components=____).fit_transform(____)
print(points.shape)

In [ ]:
couleurs = {"sante": "tab:green", "administration": "tab:blue", "education": "tab:orange"}
plt.figure(figsize=(9, 6))
for theme, c in couleurs.items():
    idx = fiches.index[fiches["theme"] == theme]
    plt.scatter(points[idx, 0], points[idx, 1], c=c, label=theme, s=80)
for i, titre in enumerate(fiches["titre"]):
    plt.annotate(titre[:22], (points[i, 0], points[i, 1]), fontsize=7)
plt.legend(); plt.title("Carte des fiches (projection 2D)"); plt.show()

## 9 · Regrouper sans étiquettes
Sans connaître les thèmes, la machine peut **regrouper** les fiches proches : c'est le *clustering*.

In [ ]:
groupes = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(emb_fiches)
fiches["groupe"] = groupes
print(pd.crosstab(fiches["groupe"], fiches["theme"]))

Les groupes retrouvent-ils les thèmes ? Quelles fiches sont mal placées, et pourquoi ?

## 10 · Les tokens des Transformers
Un Transformer ne lit pas des mots mais des **sous-mots**. Regardons comment un tokenizer découpe un texte, y compris avec des fautes et des lettres spéciales.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("xlm-roberta-base")

for texte in ["Où vacciner mon bébé ?", "ou vacsiner mon bebe", "ɖevi", "anticonstitutionnellement"]:
    print(f"{texte!r:30} →", tok.tokenize(texte))

**👉 À vous — Exercice 7.** Comptez les tokens produits par chacune des phrases ci-dessous.  
*Indice : `len(tok.tokenize(...))`.* Quelle écriture produit le plus de morceaux ? Qu'est-ce que cela implique pour un texte avec fautes ou dans une langue peu représentée ?

In [ ]:
for texte in ["Où vacciner mon bébé ?", "ou vacsiner mon bebe"]:
    print(len(____(texte)), "tokens :", texte)

## 11 · Défi : la robustesse aux fautes
Les embeddings résistent-ils mieux aux fautes que les mots entiers ? Comparez, pour une question propre et la même question abîmée, la similarité avec la bonne fiche.

In [ ]:
propre = "Où faire vacciner mon bébé ?"
abime = "ou fer vacsiner mon bebe"
fiche = fiches["document"][0]

def sim_sens(a, b):
    v = modele.encode([a, b], normalize_embeddings=True)
    return float(v[0] @ v[1])

def sim_mots(a, b):
    t = TfidfVectorizer().fit([a, b, fiche])
    m = t.transform([a, b]).toarray()
    return float(m[0] @ m[1] / (np.linalg.norm(m[0]) * np.linalg.norm(m[1]) + 1e-9))

print("Par le sens  : propre", round(sim_sens(propre, fiche), 2), "| abîmée", round(sim_sens(abime, fiche), 2))
print("Par les mots : propre", round(sim_mots(propre, fiche), 2), "| abîmée", round(sim_mots(abime, fiche), 2))

## Bilan
Discutez en binôme :
1. Sur quelles questions la recherche par le sens a-t-elle apporté le plus ? Où échoue-t-elle encore ?
2. Comment choisiriez-vous un seuil pour un vrai service public ? Qui relirait les refus ?
3. Quelles informations ne faudrait-il **jamais** envoyer à un modèle hébergé à l'extérieur ?
4. Pour une langue togolaise peu représentée, que feriez-vous avant de faire confiance à ce modèle ? *(indice : mesurer sur vos propres exemples, collecter des textes, module 3)*

**Prochain module :** adapter un modèle à vos besoins (fine-tuning, LoRA, QLoRA).